# F1 pole position predictor

## Import


In [26]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)

qualifying = pd.read_csv("F1(1950 - 2024) data\\raw\\qualifying.csv" , na_values=['\\N'])
races = pd.read_csv("F1(1950 - 2024) data\\raw\\races.csv", na_values=['\\N'])
drivers = pd.read_csv("F1(1950 - 2024) data\\raw\\drivers.csv", na_values=['\\N'])
constructors = pd.read_csv("F1(1950 - 2024) data\\raw\\constructors.csv", na_values=['\\N'])

print('Data loaded successfully')

Data loaded successfully


## Checking data

In [27]:
print(qualifying.shape)
qualifying.head()

(10494, 9)


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3
0,1,18,1,1,22,1,1:26.572,1:25.187,1:26.714
1,2,18,9,2,4,2,1:26.103,1:25.315,1:26.869
2,3,18,5,1,23,3,1:25.664,1:25.452,1:27.079
3,4,18,13,6,2,4,1:25.994,1:25.691,1:27.178
4,5,18,2,2,3,5,1:25.960,1:25.518,1:27.236


## check null

In [28]:
qualifying.isnull().sum()

qualifyId           0
raceId              0
driverId            0
constructorId       0
number              0
position            0
q1                156
q2               4647
q3               6865
dtype: int64

In [29]:
races.sample(5)

,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
138,139,2002,16,19,United States Grand Prix,2002-09-29,NaN,http://en.wikipedia.org/wiki/2002_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
490,491,1981,9,9,British Grand Prix,1981-07-18,NaN,http://en.wikipedia.org/wiki/1981_British_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
497,498,1980,1,25,Argentine Grand Prix,1980-01-13,NaN,http://en.wikipedia.org/wiki/1980_Argentine_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
219,220,1997,14,70,Austrian Grand Prix,1997-09-21,NaN,http://en.wikipedia.org/wiki/1997_Austrian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
901,904,2014,5,4,Spanish Grand Prix,2014-05-11,12:00:00,http://en.wikipedia.org/wiki/2014_Spanish_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Data manipulation

In [30]:
df = pd.merge(qualifying, races, how='left', on='raceId')
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
1455,1456,125,44,16,12,18,1:38.390,NaN,NaN,2002,2,2,Malaysian Grand Prix,2002-03-17,NaN,http://en.wikipedia.org/wiki/2002_Malaysian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3503,3505,344,807,3,10,12,1:16.770,1:16.438,NaN,2010,8,7,Canadian Grand Prix,2010-06-13,16:00:00,http://en.wikipedia.org/wiki/2010_Canadian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7963,7987,1011,154,210,8,8,1:29.688,1:29.249,1:29.015,2019,2,3,Bahrain Grand Prix,2019-03-31,15:10:00,http://en.wikipedia.org/wiki/2019_Bahrain_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1966,1967,214,56,6,6,5,1:14.860,NaN,NaN,1997,8,8,French Grand Prix,1997-06-29,NaN,http://en.wikipedia.org/wiki/1997_French_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
996,997,64,15,7,8,13,1:15.430,1:15.150,NaN,2006,12,10,German Grand Prix,2006-07-30,14:00:00,http://en.wikipedia.org/wiki/2006_German_Grand...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Drop pre 2005

In [31]:
df = df[df['year'] >= 2005]

## Convert Str to seconds

In [32]:
df['q1'].dtypes

<StringDtype(storage='python', na_value=nan)>

In [33]:
df['q1'] = df['q1'].str.split(':').str[0].astype(float) * 60 + df['q1'].str.split(':').str[1].astype(float)

df['q2'] = df['q2'].str.split(':').str[0].astype(float) * 60 + df['q2'].str.split(':').str[1].astype(float)

df['q3'] = df['q3'].str.split(':').str[0].astype(float) * 60 + df['q3'].str.split(':').str[1].astype(float)


In [34]:
df['q1'].head(10)

0    86.572
1    86.103
2    85.664
3    85.994
4    85.960
5    86.427
6    86.295
7    86.381
8    86.919
9    86.702
Name: q1, dtype: float64

## Filling nulls with per year medians

In [35]:
df['q1'] = df.groupby('year')['q1'].transform(lambda x: x.fillna(x.median()))
df['q2'] = df.groupby('year')['q2'].transform(lambda x: x.fillna(x.median()))
df['q3'] = df.groupby('year')['q3'].transform(lambda x: x.fillna(x.median()))


In [36]:
df[['q1', 'q2', 'q3']].isnull().sum()

q1      0
q2      0
q3    376
dtype: int64

## Filling q3 nulls with 0

In [37]:
df['q3'] = df['q3'].fillna(0)

## create reached_q2&q3 column

In [38]:
df['reached_q2'] = (df['q2'].notnull()).astype(int)
df['reached_q3'] = (df['q3'].notnull()).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3
520,521,42,16,12,20,21,74.122,83.1205,83.446,2007,7,19,United States Grand Prix,2007-06-17,17:00:00,http://en.wikipedia.org/wiki/2007_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
7396,7419,982,828,15,9,20,105.570,90.3070,89.480,2017,14,15,Singapore Grand Prix,2017-09-17,12:00:00,http://en.wikipedia.org/wiki/2017_Singapore_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
9641,9699,1101,846,1,4,7,102.154,101.4850,101.281,2023,4,73,Azerbaijan Grand Prix,2023-04-30,11:00:00,https://en.wikipedia.org/wiki/2023_Azerbaijan_...,2023-04-28,09:30:00,2023-04-29,09:30:00,NaN,NaN,2023-04-28,13:00:00,2023-04-29,13:30:00,1,1
6599,6622,942,832,5,55,20,127.304,93.3610,89.480,2015,16,69,United States Grand Prix,2015-10-25,19:00:00,http://en.wikipedia.org/wiki/2015_United_State...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1
3367,3369,338,15,205,18,20,89.111,91.1550,90.705,2010,2,1,Australian Grand Prix,2010-03-28,06:00:00,http://en.wikipedia.org/wiki/2010_Australian_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1


In [39]:
print(df[df['q3'].isnull()]['reached_q3'].value_counts())

Series([], Name: count, dtype: int64)


## create got_pole column

In [40]:
df['got_pole'] = (df['position'] == 1).astype(int)
df.sample(5)

,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
9044,9085,1070,846,1,4,10,77.569,77.473,96.830,2021,18,32,Mexico City Grand Prix,2021-11-07,19:00:00,http://en.wikipedia.org/wiki/2021_Mexican_Gran...,2021-11-05,NaN,2021-11-05,NaN,2021-11-06,NaN,2021-11-06,NaN,NaN,NaN,1,1,0
8235,8259,1025,1,131,44,2,93.230,93.134,92.030,2019,16,71,Russian Grand Prix,2019-09-29,11:10:00,http://en.wikipedia.org/wiki/2019_Russian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
9882,9940,1114,847,131,63,8,90.811,90.268,90.219,2023,16,22,Japanese Grand Prix,2023-09-24,05:00:00,https://en.wikipedia.org/wiki/2023_Japanese_Gr...,2023-09-22,02:30:00,2023-09-22,06:00:00,2023-09-23,02:30:00,2023-09-23,06:00:00,NaN,NaN,1,1,0
925,926,61,18,11,12,8,76.594,75.814,76.608,2006,9,7,Canadian Grand Prix,2006-06-25,13:00:00,http://en.wikipedia.org/wiki/2006_Canadian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6844,6867,956,13,3,19,10,67.419,67.145,71.977,2016,9,70,Austrian Grand Prix,2016-07-03,12:00:00,http://en.wikipedia.org/wiki/2016_Austrian_Gra...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0


In [41]:
df['got_pole'].value_counts()

got_pole
0    7872
1     394
Name: count, dtype: int64

In [42]:
df.groupby('year')['q2'].median().head(20)

year
2005    89.1590
2006    81.4840
2007    83.1205
2008    83.4555
2009    91.2220
         ...   
2020    79.9620
2021    81.0860
2022    85.2250
2023    84.4405
2024    83.8770
Name: q2, Length: 20, dtype: float64

In [43]:
for i, col in enumerate(df.columns):
    print(i, col)

pd.set_option('display.max_columns', None)
df.sample(20)

0 qualifyId
1 raceId
2 driverId
3 constructorId
4 number
5 position
6 q1
7 q2
8 q3
9 year
10 round
11 circuitId
12 name
13 date
14 time
15 url
16 fp1_date
17 fp1_time
18 fp2_date
19 fp2_time
20 fp3_date
21 fp3_time
22 quali_date
23 quali_time
24 sprint_date
25 sprint_time
26 reached_q2
27 reached_q3
28 got_pole


,qualifyId,raceId,driverId,constructorId,number,position,q1,q2,q3,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time,reached_q2,reached_q3,got_pole
7595,7619,992,154,210,8,20,88.165,88.4245,88.1940,2018,4,73,Azerbaijan Grand Prix,2018-04-29,12:10:00,http://en.wikipedia.org/wiki/2018_Azerbaijan_G...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
10353,10411,1137,842,214,10,20,103.088,103.1790,83.2340,2024,17,73,Azerbaijan Grand Prix,2024-09-15,11:00:00,https://en.wikipedia.org/wiki/2024_Azerbaijan_...,2024-09-13,09:30:00,2024-09-13,13:00:00,2024-09-14,08:30:00,2024-09-14,12:00:00,NaN,NaN,1,1,0
9473,9531,1091,848,3,23,19,116.985,85.2250,88.0995,2022,17,15,Singapore Grand Prix,2022-10-02,12:00:00,http://en.wikipedia.org/wiki/2022_Singapore_Gr...,2022-09-30,10:00:00,2022-09-30,13:00:00,2022-10-01,10:00:00,2022-10-01,13:00:00,NaN,NaN,1,1,0
1063,1064,67,14,9,14,14,82.618,82.5890,81.5425,2006,15,14,Italian Grand Prix,2006-09-10,14:00:00,http://en.wikipedia.org/wiki/2006_Italian_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6711,6734,950,832,5,55,8,97.656,97.2040,96.8810,2016,3,17,Chinese Grand Prix,2016-04-17,06:00:00,http://en.wikipedia.org/wiki/2016_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9611,9669,1099,848,3,23,17,89.994,84.4405,84.9105,2023,2,77,Saudi Arabian Grand Prix,2023-03-19,17:00:00,https://en.wikipedia.org/wiki/2023_Saudi_Arabi...,2023-03-17,13:30:00,2023-03-17,17:00:00,2023-03-18,13:30:00,2023-03-18,17:00:00,NaN,NaN,1,1,0
7496,7519,987,1,131,44,20,92.263,90.3070,89.4800,2017,19,18,Brazilian Grand Prix,2017-11-12,16:00:00,http://en.wikipedia.org/wiki/2017_Brazilian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6341,6364,928,829,209,28,19,102.091,93.3610,89.4800,2015,3,17,Chinese Grand Prix,2015-04-12,06:00:00,http://en.wikipedia.org/wiki/2015_Chinese_Gran...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0
6256,6259,917,825,1,20,7,71.134,71.2110,70.9690,2014,18,18,Brazilian Grand Prix,2014-11-09,16:00:00,http://en.wikipedia.org/wiki/2014_Brazilian_Gr...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,1,0


## Features and Target

In [44]:
features = ['q1', 'q2', 'q3', 'reached_q2', 'reached_q3', 'year', 'circuitId', 'constructorId', 'driverId', 'round']
target = 'got_pole'

## New df

In [46]:
df_cleaned = df[features + [target]]
print(df_cleaned.shape)
df_cleaned.sample(5)

(8266, 11)


,q1,q2,q3,reached_q2,reached_q3,year,circuitId,constructorId,driverId,round,got_pole
4566,97.210,96.642,96.3240,1,1,2011,2,4,808,2,0
3508,77.611,77.384,90.7050,1,1,2010,7,15,37,8,0
3414,99.278,91.155,90.7050,1,1,2010,17,166,10,4,0
358,72.348,72.137,86.7915,1,1,2008,18,4,12,18,0
5308,93.349,92.469,95.6570,1,1,2012,22,131,30,15,0
